# 03_test_resshift_seeds_t4

Seed experiment: ResShift on the 1,104 test inputs with seed offsets 1 to 4 (offset 0 is the main run of notebook 02), so 5 seeds in total. If the GPU time runs out, set SPLIT = 'dev' in the settings cell and say so in the report.

**Runtime:** free Google Colab or Kaggle with an NVIDIA T4, about 70 minutes. Kaggle needs *Internet on* (weights and the official ResShift code are downloaded).

**Before running:** create `data/gpu_inputs_test.zip` locally with `python scripts/make_gpu_inputs.py --split test`.
Colab: run the first cell and upload it when asked (or put it at `/content/gpu_inputs_test.zip`). Kaggle: add it as a dataset with any name; the notebook finds the zip, or the folder Kaggle unpacks it into, under `/kaggle/input`.

**After running:** download `03_test_resshift_seeds_t4_results.zip` (Colab downloads it; Kaggle shows a download link under the last cell, and the zip is the only file in Output) and unzip it at the repository root.
It contains `outputs/test/<method>/...` (PNG outputs with `manifest.json`) and `results/runtime/...` (timing, peak memory, environment).
Then run the local commands in README.md, section *Run experiments*.

The cells below embed the repository files verbatim; each file is checked against its SHA-256 from the build. Rebuild with `python scripts/build_notebooks.py` after changing them.


In [ ]:
SPLIT = 'test'
METHODS = ['resshift']
SEED_OFFSETS = [1, 2, 3, 4]
NOTEBOOK = '03_test_resshift_seeds_t4'
INPUT_ARCHIVE = 'gpu_inputs_test.zip'
INPUTS_SHA256_JSON = '5b62afc2cd75dc0084a978f043f958aefa2b16465a5b72e98c238a6ec586a3b3'   # sha256 of data/processed/inputs_sha256.json at build time


In [ ]:
import glob
import hashlib
import os
import subprocess
import sys
import zipfile
from pathlib import Path

ON_KAGGLE = Path('/kaggle/working').exists()
WORK = Path('/tmp/sr') if ON_KAGGLE else Path('/content/sr')   # outside /kaggle/working: the results zip is the only Kaggle output
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

if ON_KAGGLE:
    # Kaggle usually unpacks an uploaded zip: accept the archive itself or its unpacked data/processed/ folder.
    found = glob.glob(f'/kaggle/input/**/{INPUT_ARCHIVE}', recursive=True)
    unpacked = glob.glob('/kaggle/input/**/data/processed/inputs_sha256.json', recursive=True)
    assert found or unpacked, f'add {INPUT_ARCHIVE} as a Kaggle dataset (the dataset name does not matter)'
    archive = Path(found[0]) if found else Path(unpacked[0]).parents[2]
else:
    archive = Path('/content') / INPUT_ARCHIVE
    if not archive.exists():
        from google.colab import files
        uploaded = files.upload()
        archive = Path('/content') / next(iter(uploaded))
if archive.is_file():
    print('input archive:', archive, hashlib.sha256(archive.read_bytes()).hexdigest())
else:
    print('unpacked inputs:', archive)

import torch
assert torch.cuda.is_available(), 'select a GPU runtime (T4)'
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)
if 'T4' not in torch.cuda.get_device_name(0):
    print('WARNING: not a T4; runtime numbers will not be comparable')


In [ ]:
FILES = {
"paths.py": "\"\"\"Where the data, weights, outputs and results live.\n\nResolution order everywhere: environment variable > YAML value > default below.\nThe YAML file is the one passed with --config to a root entry point, or $SR_CONFIG, or configs/default.yaml.\n\nSR_DATA        - data root: raw/ (downloads) and processed/ (hr/, lr/, manifest.csv)\nSR_WEIGHTS     - flat folder with the downloaded weights listed in scripts/SHA256SUMS\nSR_OUTPUTS     - super-resolution outputs: <split>/<method>/jpeg_<quality>/<image_id>.png plus manifest.json\nSR_RESULTS     - tables, JSON reports and figures\nSR_THIRD_PARTY - pinned third-party source code (ResShift)\nSR_CACHE       - model caches of the metric libraries (pyiqa, insightface)\n\"\"\"\n\nimport hashlib\nimport os\nfrom pathlib import Path\n\nimport yaml\n\nROOT = Path(__file__).resolve().parent\nDEFAULT_CONFIG = ROOT / \"configs\" / \"default.yaml\"\n\nDEFAULTS = {\n    \"data\": ROOT / \"data\",\n    \"weights\": ROOT / \"models\",\n    \"outputs\": ROOT / \"outputs\",\n    \"results\": ROOT / \"results\",\n    \"third_party\": ROOT / \"third_party\",\n    \"cache\": ROOT / \"data\" / \"cache\",\n}\nENV_NAMES = {\n    \"data\": \"SR_DATA\",\n    \"weights\": \"SR_WEIGHTS\",\n    \"outputs\": \"SR_OUTPUTS\",\n    \"results\": \"SR_RESULTS\",\n    \"third_party\": \"SR_THIRD_PARTY\",\n    \"cache\": \"SR_CACHE\",\n}\n\n\ndef config_path():\n    return Path(os.environ.get(\"SR_CONFIG\", DEFAULT_CONFIG))\n\n\ndef load_config(path=None):\n    with open(path or config_path()) as handle:\n        return yaml.safe_load(handle)\n\n\ndef resolve(key, cfg=None):\n    \"\"\"Environment variable, then the YAML value under paths:, then the default. Relative YAML paths are relative to ROOT.\"\"\"\n    if os.environ.get(ENV_NAMES[key]):\n        return Path(os.environ[ENV_NAMES[key]])\n    cfg = load_config() if cfg is None else cfg\n    value = (cfg.get(\"paths\") or {}).get(key)\n    if value:\n        value = Path(value)\n        return value if value.is_absolute() else ROOT / value\n    return DEFAULTS[key]\n\n\ndef use_config(path):\n    \"\"\"Called by the root entry points before they import code/: every module then resolves the same file.\"\"\"\n    os.environ[\"SR_CONFIG\"] = str(Path(path).resolve())\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with open(path, \"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1 << 22), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef check_frozen(config, split):\n    \"\"\"The test split runs only with configs/final.yaml whose SHA-256 matches configs/final.yaml.sha256.\"\"\"\n    if split != \"test\":\n        return\n    config = Path(config).resolve()\n    digest_file = config.with_name(config.name + \".sha256\")\n    if config.name != \"final.yaml\" or not digest_file.exists():\n        raise SystemExit(\"the test split runs only with --config configs/final.yaml (frozen, with final.yaml.sha256)\")\n    expected = digest_file.read_text().split()[0]\n    if sha256_file(config) != expected:\n        raise SystemExit(f\"{config} was changed after it was frozen: {sha256_file(config)[:12]} != {expected[:12]}\")\n\n\ndef run_stage(script, args, config):\n    \"\"\"Run one script of code/ with the given config; the root entry points are thin wrappers around this.\"\"\"\n    import subprocess\n    import sys\n\n    import datetime\n    import json\n    import platform\n    import time\n\n    environment = dict(os.environ, SR_CONFIG=str(Path(config).resolve()))\n    start = time.perf_counter()\n    subprocess.run([sys.executable, str(ROOT / script), *map(str, args)], env=environment, check=True, cwd=ROOT)\n    log = resolve(\"results\", load_config(config)) / \"runtime\" / \"stage_times.json\"\n    log.parent.mkdir(parents=True, exist_ok=True)\n    entries = json.loads(log.read_text()) if log.exists() else []\n    entries.append({\"script\": script, \"args\": list(map(str, args)), \"config\": Path(config).name, \"seconds\": time.perf_counter() - start,\n                    \"machine\": f\"{platform.system()} {platform.machine()} {platform.processor()}\",\n                    \"cache\": str(resolve(\"cache\", load_config(config))), \"finished\": datetime.datetime.now().isoformat(timespec=\"seconds\")})\n    log.write_text(json.dumps(entries, indent=1))\n\n\n_CFG = load_config()\nDATA_ROOT = resolve(\"data\", _CFG)\nWEIGHTS_ROOT = resolve(\"weights\", _CFG)\nOUTPUTS_ROOT = resolve(\"outputs\", _CFG)\nRESULTS_ROOT = resolve(\"results\", _CFG)\nTHIRD_PARTY_ROOT = resolve(\"third_party\", _CFG)\nCACHE_ROOT = resolve(\"cache\", _CFG)\n",
"inference.py": "\"\"\"Stage 2, inference: x4 super-resolution of every low-resolution input of a split with the configured methods.\n\n    python inference.py --config configs/default.yaml --split dev\n    python inference.py --config configs/final.yaml --split test                                   # once, frozen settings\n    python inference.py --config configs/final.yaml --split test --methods resshift --seed-offsets 1 2 3 4\n\nOutputs: outputs/<split>/<method>/jpeg_<q>/<image_id>.png with manifest.json (weights, code, settings and input hashes);\nan output folder whose manifest matches is skipped. Timing and peak GPU memory: results/runtime/.\n\"\"\"\n\nimport argparse\n\nimport paths\n\nparser = argparse.ArgumentParser(description=\"Super-resolution inference\")\nparser.add_argument(\"--config\", default=\"configs/default.yaml\")\nparser.add_argument(\"--split\", default=\"dev\", choices=[\"dev\", \"test\"])\nparser.add_argument(\"--methods\", nargs=\"+\", default=None)\nparser.add_argument(\"--seed-offsets\", nargs=\"+\", type=int, default=None)\nparser.add_argument(\"--force\", action=\"store_true\")\nargs = parser.parse_args()\npaths.check_frozen(args.config, args.split)\ncfg = paths.load_config(args.config)\nextra = [\"--methods\", *(args.methods or cfg[\"methods\"]), \"--seed-offsets\", *(args.seed_offsets or [cfg[\"resshift\"][\"seed_offset\"]])]\npaths.run_stage(\"code/models/upscale.py\", [\"--split\", args.split, *extra, *([\"--force\"] if args.force else [])], args.config)\n",
"configs/default.yaml": "# Working settings for the dev split. Values marked \"selected on dev\" are filled in by\n# code/evaluation/select_settings.py and copied into configs/final.yaml, which is frozen before the test run.\n\npaths:                       # environment variables SR_DATA, SR_WEIGHTS, ... override these (see paths.py)\n  data: data\n  weights: models\n  outputs: outputs\n  results: results\n  third_party: third_party\n  cache: data/cache\n\ndata:                        # must match the constants in code/datasets/prepare_data.py (checked at run time)\n  seed: 42\n  hr_size: 512\n  scale: 4\n  jpeg_qualities: [none, 50, 30, 10]\n  jpeg_subsampling: 2        # Pillow code for 4:2:0\n\nmethods: [bicubic, realesrnet, realesrgan, resshift]\n\nrrdbnet:\n  weights: {realesrnet: RealESRNet_x4plus.pth, realesrgan: RealESRGAN_x4plus.pth}\n  precision: float32\n  batch_size: 1\n  tiling: none\n\nresshift:\n  version: v3\n  official_commit: bb03b7d21614cace01787e097c8a6ab6b945227d\n  model_weights: resshift_realsrx4_s4_v3.pth\n  autoencoder_weights: autoencoder_vq_f4.pth\n  steps: 4                   # fixed by the v3 checkpoint\n  precision: float32 weights, float16 autocast\n  amp: true\n  seed_policy: (crc32(image_id) + offset) & 0x7fffffff, the same seed for every JPEG level\n  seed_offset: 0             # offset 0 is the stage 4 seed policy\n  seed_offsets: [0, 1, 2, 3, 4]\n\nruntime:\n  warmup_runs: 3\n  bicubic_device: cpu\n\nmetrics:\n  psnr_ssim: {channel: Y (BT.601), crop_border: 4}\n  lpips: {net: alex, version: \"0.1\"}\n  musiq:\n    library: pyiqa\n    metric: musiq            # pyiqa default for musiq: the KonIQ-10k checkpoint\n    device: cpu\n    checkpoint: null         # file name and SHA-256 are recorded by select_settings.py\n  ocr:\n    engine: tesseract\n    lang: eng\n    normalisation: collapse every whitespace run to one space, strip; case-sensitive\n    cer: Levenshtein distance / reference length, not capped\n    candidates:\n      psm: [3, 4, 6]\n      oem: [1, 3]\n      preprocess: [none, otsu, upscale2_otsu]\n    selected: null           # selected on dev HR images: lowest mean CER, ties broken by list order\n  arcface:\n    library: insightface\n    model_pack: buffalo_l    # SCRFD-10GF detector + ResNet-50 ArcFace (WebFace600K)\n    det_size_candidates: [512, 640, 320]   # selected on dev: fewest HR detection failures, ties by this order\n    det_size: null\n    det_thresh: 0.5\n    face: largest detected face\n    false_match_rate: 0.01\n    threshold: null          # selected on dev: impostor pairs (HR vs HR, different faces), quantile 0.99 (method \"higher\")\n    match: similarity > threshold\n    detection_failure: similarity missing (excluded from the mean, counted); counted as a non-match in the match rate\n\nstatistics:\n  bootstrap_samples: 10000\n  seed: 42\n  confidence: 0.95\n  interval: percentile endpoints [lo, hi]\n  correction: none (pointwise intervals)\n\n# Analyses, written before the test run. \"lo > 0\" means the lower end of the 95% interval is above zero.\nanalysis:\n  generative: [realesrgan, resshift]\n  reference: realesrnet          # non-generative network with the same architecture and data as Real-ESRGAN\n  jpeg_hypothesis:\n    statement: all models degrade sharply below JPEG quality 30\n    per_image: drop_low = M(30) - M(10), drop_high = M(50) - M(30), signed so that a positive drop is a loss (LPIPS sign flipped)\n    statistic: mean of (drop_low - drop_high) over images, paired percentile bootstrap\n    metrics: [psnr, ssim, lpips]\n    verdict: supported if lo > 0 for all 3 metrics; not supported if lo > 0 for none; mixed otherwise\n    unit: each method x content type\n  hypothesis_checks:\n    sharper_but_lower_psnr: generative - reference, JPEG none and 10; LPIPS hi < 0, MUSIQ lo > 0, PSNR hi < 0\n    invented_letters: generative - reference on text, JPEG none and 10, also by HR font size; CER lo > 0 while LPIPS hi < 0\n    identity_change: generative - reference on faces, JPEG none and 10; ArcFace similarity hi < 0; match rate reported\n    jpeg_below_30: see jpeg_hypothesis\n  failure_cases:\n    a: text, any JPEG level; LPIPS ranks a generative method best of the 4 and its CER is the worst of the 4; take the case with the largest CER gap to the best-CER method\n    b: faces, JPEG none; the lowest ArcFace similarity of any generative method (detection failures are listed separately)\n    c: JPEG 10, any content; the largest PSNR deficit of a generative method below bicubic\n    d: ResShift, HR font size 16, JPEG none; the highest ResShift CER, ties by the largest CER gap to Real-ESRNet\n    crop: 128x128 HR pixels centred on the maximum of the 31x31 box-filtered absolute Y error of the failing method\n    fallback: if no image meets rule a, report \"no case\" and keep b, c and d\n",
"configs/final.yaml": "# Frozen settings for the final test run. Do not edit: the SHA-256 is in final.yaml.sha256 and is checked before every test run.\npaths:\n  data: data\n  weights: models\n  outputs: outputs\n  results: results\n  third_party: third_party\n  cache: data/cache\ndata:\n  seed: 42\n  hr_size: 512\n  scale: 4\n  jpeg_qualities:\n  - none\n  - 50\n  - 30\n  - 10\n  jpeg_subsampling: 2\nmethods:\n- bicubic\n- realesrnet\n- realesrgan\n- resshift\nrrdbnet:\n  weights:\n    realesrnet: RealESRNet_x4plus.pth\n    realesrgan: RealESRGAN_x4plus.pth\n  precision: float32\n  batch_size: 1\n  tiling: none\nresshift:\n  version: v3\n  official_commit: bb03b7d21614cace01787e097c8a6ab6b945227d\n  model_weights: resshift_realsrx4_s4_v3.pth\n  autoencoder_weights: autoencoder_vq_f4.pth\n  steps: 4\n  precision: float32 weights, float16 autocast\n  amp: true\n  seed_policy: (crc32(image_id) + offset) & 0x7fffffff, the same seed for every JPEG level\n  seed_offset: 0\n  seed_offsets:\n  - 0\n  - 1\n  - 2\n  - 3\n  - 4\nruntime:\n  warmup_runs: 3\n  bicubic_device: cpu\nmetrics:\n  psnr_ssim:\n    channel: Y (BT.601)\n    crop_border: 4\n  lpips:\n    net: alex\n    version: '0.1'\n  musiq:\n    library: pyiqa\n    metric: musiq\n    device: cpu\n    checkpoint:\n      file: musiq_koniq_ckpt-e95806b9.pth\n      sha256: e95806b9eae5f3814c410f574ba8e552362bd5bc63d758ed5b97860f5d6185aa\n  ocr:\n    engine: tesseract\n    lang: eng\n    normalisation: collapse every whitespace run to one space, strip; case-sensitive\n    cer: Levenshtein distance / reference length, not capped\n    candidates:\n      psm:\n      - 3\n      - 4\n      - 6\n      oem:\n      - 1\n      - 3\n      preprocess:\n      - none\n      - otsu\n      - upscale2_otsu\n    selected:\n      psm: 4\n      oem: 1\n      preprocess: none\n    tesseract: tesseract 5.5.1\n    tessdata_sha256: 7d4322bd2a7749724879683fc3912cb542f19906c83bcc1a52132556427170b2\n  arcface:\n    library: insightface\n    model_pack: buffalo_l\n    det_size_candidates:\n    - 512\n    - 640\n    - 320\n    det_size: 512\n    det_thresh: 0.5\n    face: largest detected face\n    false_match_rate: 0.01\n    threshold: 0.1819307956740638\n    match: similarity > threshold\n    detection_failure: similarity missing (excluded from the mean, counted); counted as a non-match in the match rate\n    models_sha256:\n      1k3d68.onnx: df5c06b8a0c12e422b2ed8947b8869faa4105387f199c477af038aa01f9a45cc\n      2d106det.onnx: f001b856447c413801ef5c42091ed0cd516fcd21f2d6b79635b1e733a7109dbf\n      det_10g.onnx: 5838f7fe053675b1c7a08b633df49e7af5495cee0493c7dcf6697200b85b5b91\n      genderage.onnx: 4fde69b1c810857b88c64a335084f1c3fe8f01246c9a191b48c7bb756d6652fb\n      w600k_r50.onnx: 4c06341c33c2ca1f86781dab0e829f88ad5b64be9fba56e56bc9ebdefc619e43\nstatistics:\n  bootstrap_samples: 10000\n  seed: 42\n  confidence: 0.95\n  interval: percentile endpoints [lo, hi]\n  correction: none (pointwise intervals)\nanalysis:\n  generative:\n  - realesrgan\n  - resshift\n  reference: realesrnet\n  jpeg_hypothesis:\n    statement: all models degrade sharply below JPEG quality 30\n    per_image: drop_low = M(30) - M(10), drop_high = M(50) - M(30), signed so that a positive drop is a loss (LPIPS sign flipped)\n    statistic: mean of (drop_low - drop_high) over images, paired percentile bootstrap\n    metrics:\n    - psnr\n    - ssim\n    - lpips\n    verdict: supported if lo > 0 for all 3 metrics; not supported if lo > 0 for none; mixed otherwise\n    unit: each method x content type\n  hypothesis_checks:\n    sharper_but_lower_psnr: generative - reference, JPEG none and 10; LPIPS hi < 0, MUSIQ lo > 0, PSNR hi < 0\n    invented_letters: generative - reference on text, JPEG none and 10, also by HR font size; CER lo > 0 while LPIPS hi < 0\n    identity_change: generative - reference on faces, JPEG none and 10; ArcFace similarity hi < 0; match rate reported\n    jpeg_below_30: see jpeg_hypothesis\n  failure_cases:\n    a: text, any JPEG level; LPIPS ranks a generative method best of the 4 and its CER is the worst of the 4; take the case with the largest CER\n      gap to the best-CER method\n    b: faces, JPEG none; the lowest ArcFace similarity of any generative method (detection failures are listed separately)\n    c: JPEG 10, any content; the largest PSNR deficit of a generative method below bicubic\n    d: ResShift, HR font size 16, JPEG none; the highest ResShift CER, ties by the largest CER gap to Real-ESRNet\n    crop: 128x128 HR pixels centred on the maximum of the 31x31 box-filtered absolute Y error of the failing method\n    fallback: if no image meets rule a, report \"no case\" and keep b, c and d\nfrozen:\n  date: '2026-11-01'\n  selected_on: dev split only (64 images); see results/analysis/*_dev.json\n  test_split_note: an exploratory all-method run on test exists from cc9a873 (results/legacy_colab/); none of its numbers were used to choose\n    any setting\n",
"configs/final.yaml.sha256": "70f6c341e963a068e901be19d9dc11f36cc5df078ef04dfaf9df0846abbb1059  final.yaml\n",
"scripts/SHA256SUMS": "# SHA-256 of the downloaded weights (download_data.py --only weights verifies them).\n# ResShift and autoencoder: identical to results/resshift_dev/settings.json (stage 4 T4 run).\n# RealESRNet / RealESRGAN: official v0.1.1 / v0.1.0 release files, first recorded on 2026-11-01.\na820b9bde89a874d7599d545567308ce6c128fc8754a53208eda016d40aa81df  RealESRNet_x4plus.pth\n4fa0d38905f75ac06eb49a7951b426670021be3018265fd191d2125df9d682f1  RealESRGAN_x4plus.pth\n878da888bbbe10b8b0d4ff179c2de3059fd8a4cf2c705942a77a7f3b23ea1efd  resshift_realsrx4_s4_v3.pth\ne967eaa2c380152811c53e312990d6967a8a957c4f7590f19490ec72110b2889  autoencoder_vq_f4.pth\n",
"code/datasets/download_data.py": "import argparse\nimport random\nimport sys\nimport zipfile\nfrom pathlib import Path\n\nimport requests\nfrom tqdm import tqdm\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nsys.path.append(str(PROJECT_ROOT))\nimport paths\n\nRAW_DIR = paths.DATA_ROOT / \"raw\"\nMODELS_DIR = paths.WEIGHTS_ROOT\nSHA256SUMS = PROJECT_ROOT / \"scripts\" / \"SHA256SUMS\"\n\nDIV2K_URL = \"https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip\"\nDIV2K_DIR = RAW_DIR / \"DIV2K_valid_HR\"\n\nFFHQ_URL = \"https://huggingface.co/datasets/marcosv/ffhq-dataset/resolve/main/Part1/{:05d}.png\"\nFFHQ_DIR = RAW_DIR / \"ffhq\"\nFFHQ_POOL = range(0, 1000)\nFACE_COUNT = 120\n\nWEIGHTS = {\n    \"RealESRNet_x4plus.pth\": \"https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth\",\n    \"RealESRGAN_x4plus.pth\": \"https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth\",\n    \"resshift_realsrx4_s4_v3.pth\": \"https://github.com/zsyOAOA/ResShift/releases/download/v2.0/resshift_realsrx4_s4_v3.pth\",\n    \"autoencoder_vq_f4.pth\": \"https://github.com/zsyOAOA/ResShift/releases/download/v2.0/autoencoder_vq_f4.pth\",\n}\n\n\ndef download_file(url, path):\n    if path.exists():\n        return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    partial_path = path.with_name(path.name + \".part\")\n    response = requests.get(url, stream=True, timeout=60)\n    response.raise_for_status()\n    total_bytes = int(response.headers.get(\"content-length\", 0))\n    with open(partial_path, \"wb\") as file, tqdm(total=total_bytes, unit=\"B\", unit_scale=True, desc=path.name) as bar:\n        for chunk in response.iter_content(chunk_size=1 << 20):\n            file.write(chunk)\n            bar.update(len(chunk))\n    partial_path.rename(path)\n\n\ndef download_div2k():\n    if DIV2K_DIR.exists() and len(list(DIV2K_DIR.glob(\"*.png\"))) == 100:\n        return\n    archive_path = RAW_DIR / \"DIV2K_valid_HR.zip\"\n    download_file(DIV2K_URL, archive_path)\n    with zipfile.ZipFile(archive_path) as archive:\n        archive.extractall(RAW_DIR)\n    archive_path.unlink()\n\n\ndef choose_face_ids(seed):\n    generator = random.Random(seed)\n    return sorted(generator.sample(list(FFHQ_POOL), FACE_COUNT))\n\n\ndef download_faces(seed):\n    for face_id in tqdm(choose_face_ids(seed), desc=\"FFHQ\"):\n        download_file(FFHQ_URL.format(face_id), FFHQ_DIR / f\"{face_id:05d}.png\")\n\n\ndef read_checksums(path=SHA256SUMS):\n    checksums = {}\n    for line in path.read_text().splitlines():\n        if line.strip() and not line.startswith(\"#\"):\n            digest, name = line.split()\n            checksums[name] = digest\n    return checksums\n\n\ndef verify_weights(names=None):\n    checksums = read_checksums()\n    for name in names or WEIGHTS:\n        actual = paths.sha256_file(MODELS_DIR / name)\n        if actual != checksums[name]:\n            raise SystemExit(f\"checksum mismatch for {MODELS_DIR / name}: {actual} != {checksums[name]} (scripts/SHA256SUMS)\")\n        print(f\"{name}: OK\")\n\n\ndef download_weights(names=None):\n    for name in names or WEIGHTS:\n        download_file(WEIGHTS[name], MODELS_DIR / name)\n    verify_weights(names)\n\n\ndef main(seed=42, only=(\"data\", \"weights\")):\n    if \"data\" in only:\n        download_div2k()\n        download_faces(seed)\n    if \"weights\" in only:\n        download_weights()\n    print(f\"DIV2K images: {len(list(DIV2K_DIR.glob('*.png')))}\")\n    print(f\"FFHQ images: {len(list(FFHQ_DIR.glob('*.png')))}\")\n    print(f\"Weights: {sorted(path.name for path in MODELS_DIR.glob('*.pth'))}\")\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--seed\", type=int, default=42)\n    parser.add_argument(\"--only\", nargs=\"+\", default=[\"data\", \"weights\"], choices=[\"data\", \"weights\"])\n    args = parser.parse_args()\n    main(args.seed, args.only)\n",
"code/datasets/prepare_data.py": "import argparse\nimport io\nimport json\nimport random\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\n\nfrom download_data import DIV2K_DIR, FFHQ_DIR\nfrom render_text import render_text_set\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nsys.path.append(str(PROJECT_ROOT))\nimport paths\n\nPROCESSED_DIR = paths.DATA_ROOT / \"processed\"\nHR_DIR = PROCESSED_DIR / \"hr\"\nLR_DIR = PROCESSED_DIR / \"lr\"\n\nHR_SIZE = 512\nSCALE = 4\nJPEG_QUALITIES = [\"none\", 50, 30, 10]\nJPEG_SUBSAMPLING = 2\n\nOBJECT_DEV_COUNT = 20\nFACE_DEV_COUNT = 20\nTEXT_DEV_REPEATS = 1\nTEXT_TEST_REPEATS = 4\nTEXT_DEV_SEED = 2002\nTEXT_TEST_SEED = 1001\n\n\ndef crop_center(image, size=HR_SIZE):\n    left = (image.width - size) // 2\n    top = (image.height - size) // 2\n    return image.crop((left, top, left + size, top + size))\n\n\ndef split_sources(source_paths, dev_count, seed):\n    source_paths = sorted(source_paths)\n    generator = random.Random(seed)\n    dev_paths = set(generator.sample(source_paths, dev_count))\n    return {path: \"dev\" if path in dev_paths else \"test\" for path in source_paths}\n\n\ndef build_objects(seed):\n    rows = []\n    splits = split_sources(DIV2K_DIR.glob(\"*.png\"), OBJECT_DEV_COUNT, seed)\n    for path, split in splits.items():\n        image_id = f\"object_{path.stem}\"\n        image = crop_center(Image.open(path).convert(\"RGB\"))\n        image.save(HR_DIR / f\"{image_id}.png\")\n        rows.append({\"image_id\": image_id, \"content\": \"objects\", \"split\": split, \"source\": f\"DIV2K/{path.name}\"})\n    return rows\n\n\ndef build_faces(seed):\n    rows = []\n    splits = split_sources(FFHQ_DIR.glob(\"*.png\"), FACE_DEV_COUNT, seed)\n    for path, split in splits.items():\n        image_id = f\"face_{path.stem}\"\n        image = Image.open(path).convert(\"RGB\").resize((HR_SIZE, HR_SIZE), Image.BICUBIC)\n        image.save(HR_DIR / f\"{image_id}.png\")\n        rows.append({\"image_id\": image_id, \"content\": \"faces\", \"split\": split, \"source\": f\"FFHQ/{path.name}\"})\n    return rows\n\n\ndef build_text():\n    rows = []\n    text_sets = {\n        \"dev\": render_text_set(TEXT_DEV_REPEATS, TEXT_DEV_SEED),\n        \"test\": render_text_set(TEXT_TEST_REPEATS, TEXT_TEST_SEED),\n    }\n    for split, samples in text_sets.items():\n        for index, sample in enumerate(samples):\n            image_id = f\"text_{split}_{index:03d}\"\n            sample[\"image\"].save(HR_DIR / f\"{image_id}.png\")\n            rows.append({\n                \"image_id\": image_id,\n                \"content\": \"text\",\n                \"split\": split,\n                \"source\": f\"rendered/{image_id}\",\n                \"font\": sample[\"font\"],\n                \"font_size\": sample[\"font_size\"],\n                \"kind\": sample[\"kind\"],\n                \"text\": sample[\"text\"],\n            })\n    return rows\n\n\ndef compress_jpeg(image, quality):\n    buffer = io.BytesIO()\n    image.save(buffer, format=\"JPEG\", quality=quality, subsampling=JPEG_SUBSAMPLING)\n    buffer.seek(0)\n    return Image.open(buffer).convert(\"RGB\")\n\n\ndef make_low_resolution(image_id):\n    high_resolution = Image.open(HR_DIR / f\"{image_id}.png\")\n    low_size = HR_SIZE // SCALE\n    low_resolution = high_resolution.resize((low_size, low_size), Image.BICUBIC)\n    for quality in JPEG_QUALITIES:\n        degraded = low_resolution if quality == \"none\" else compress_jpeg(low_resolution, quality)\n        degraded.save(LR_DIR / f\"jpeg_{quality}\" / f\"{image_id}.png\")\n\n\ndef check_leakage(manifest):\n    dev = manifest[manifest[\"split\"] == \"dev\"]\n    test = manifest[manifest[\"split\"] == \"test\"]\n    shared_sources = set(dev[\"source\"]) & set(test[\"source\"])\n    shared_texts = set(dev[\"text\"].dropna()) & set(test[\"text\"].dropna())\n    assert not shared_sources, f\"sources in both splits: {shared_sources}\"\n    assert not shared_texts, \"identical text in dev and test\"\n    return {\"shared_sources\": len(shared_sources), \"shared_texts\": len(shared_texts)}\n\n\ndef check_config(cfg):\n    expected = {\"hr_size\": HR_SIZE, \"scale\": SCALE, \"jpeg_qualities\": JPEG_QUALITIES, \"jpeg_subsampling\": JPEG_SUBSAMPLING}\n    for key, value in expected.items():\n        if cfg[\"data\"][key] != value:\n            raise SystemExit(f\"config data.{key} = {cfg['data'][key]!r}, but prepare_data.py uses {value!r}\")\n\n\ndef input_hashes(manifest):\n    \"\"\"SHA-256 of every ground-truth and low-resolution file; output manifests are keyed by these hashes.\"\"\"\n    hashes = {}\n    for image_id in manifest[\"image_id\"]:\n        hashes[f\"hr/{image_id}.png\"] = paths.sha256_file(HR_DIR / f\"{image_id}.png\")\n        for quality in JPEG_QUALITIES:\n            name = f\"lr/jpeg_{quality}/{image_id}.png\"\n            hashes[name] = paths.sha256_file(PROCESSED_DIR / name)\n    return hashes\n\n\ndef main(seed=42):\n    check_config(paths.load_config())\n    HR_DIR.mkdir(parents=True, exist_ok=True)\n    for quality in JPEG_QUALITIES:\n        (LR_DIR / f\"jpeg_{quality}\").mkdir(parents=True, exist_ok=True)\n\n    rows = build_objects(seed) + build_faces(seed) + build_text()\n    manifest = pd.DataFrame(rows)\n    for image_id in manifest[\"image_id\"]:\n        make_low_resolution(image_id)\n    manifest.to_csv(PROCESSED_DIR / \"manifest.csv\", index=False)\n    (PROCESSED_DIR / \"inputs_sha256.json\").write_text(json.dumps(input_hashes(manifest), indent=1))\n\n    counts = manifest.groupby([\"content\", \"split\"]).size().unstack().to_dict(orient=\"index\")\n    report = {\n        \"images\": len(manifest),\n        \"images_per_content_and_split\": counts,\n        \"hr_size\": HR_SIZE,\n        \"lr_size\": HR_SIZE // SCALE,\n        \"jpeg_qualities\": JPEG_QUALITIES,\n        \"lr_images\": len(manifest) * len(JPEG_QUALITIES),\n        \"leakage_check\": check_leakage(manifest),\n        \"split\": \"by source image; dev and test text rendered with different seeds\",\n        \"seed\": seed,\n    }\n    report_json = json.dumps(report, indent=2)\n    (PROCESSED_DIR / \"data_report.json\").write_text(report_json)\n    print(report_json)\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--seed\", type=int, default=42)\n    args = parser.parse_args()\n    main(args.seed)\n",
"code/datasets/render_text.py": "import random\nimport string\nfrom pathlib import Path\n\nimport matplotlib\nfrom PIL import Image, ImageDraw, ImageFont\n\nFONT_DIR = Path(matplotlib.get_data_path()) / \"fonts\" / \"ttf\"\nFONTS = [\"DejaVuSans.ttf\", \"DejaVuSerif.ttf\", \"DejaVuSansMono.ttf\"]\nFONT_SIZES = [16, 24, 32, 48]\nKINDS = [\"words\", \"codes\"]\n\nIMAGE_SIZE = 512\nMARGIN = 24\nLINE_SPACING = 1.6\nCODE_LENGTH = 6\nCODE_ALPHABET = string.ascii_uppercase + string.digits\n\nWORDS = [\n    \"about\", \"above\", \"after\", \"again\", \"always\", \"animal\", \"answer\", \"apple\", \"around\", \"autumn\",\n    \"become\", \"before\", \"begin\", \"better\", \"between\", \"bridge\", \"bright\", \"bring\", \"build\", \"camera\",\n    \"carry\", \"castle\", \"center\", \"change\", \"children\", \"circle\", \"city\", \"clean\", \"close\", \"cloud\",\n    \"color\", \"common\", \"corner\", \"country\", \"course\", \"cover\", \"dance\", \"decide\", \"deep\", \"dinner\",\n    \"doctor\", \"double\", \"dream\", \"drive\", \"early\", \"earth\", \"east\", \"eight\", \"energy\", \"enough\",\n    \"evening\", \"every\", \"example\", \"family\", \"famous\", \"father\", \"field\", \"figure\", \"final\", \"finger\",\n    \"first\", \"flower\", \"follow\", \"forest\", \"friend\", \"front\", \"garden\", \"glass\", \"golden\", \"great\",\n    \"green\", \"ground\", \"group\", \"happy\", \"heavy\", \"history\", \"horse\", \"house\", \"hundred\", \"idea\",\n    \"island\", \"journey\", \"kitchen\", \"known\", \"language\", \"large\", \"later\", \"learn\", \"letter\", \"light\",\n    \"listen\", \"little\", \"machine\", \"market\", \"matter\", \"measure\", \"middle\", \"minute\", \"modern\", \"money\",\n    \"morning\", \"mother\", \"mountain\", \"music\", \"nature\", \"never\", \"night\", \"north\", \"notice\", \"number\",\n    \"ocean\", \"often\", \"orange\", \"order\", \"other\", \"paper\", \"people\", \"person\", \"picture\", \"place\",\n    \"plant\", \"point\", \"power\", \"problem\", \"public\", \"question\", \"quick\", \"quiet\", \"reach\", \"ready\",\n    \"reason\", \"record\", \"river\", \"round\", \"school\", \"science\", \"second\", \"seven\", \"shape\", \"short\",\n    \"silver\", \"simple\", \"small\", \"sound\", \"south\", \"space\", \"spring\", \"square\", \"start\", \"story\",\n    \"street\", \"strong\", \"summer\", \"system\", \"table\", \"thing\", \"think\", \"thousand\", \"three\", \"today\",\n    \"together\", \"travel\", \"under\", \"until\", \"village\", \"voice\", \"water\", \"weather\", \"white\", \"window\",\n    \"winter\", \"without\", \"wonder\", \"world\", \"write\", \"yellow\", \"young\",\n]\n\n\ndef make_token(kind, generator):\n    if kind == \"words\":\n        return generator.choice(WORDS)\n    return \"\".join(generator.choice(CODE_ALPHABET) for _ in range(CODE_LENGTH))\n\n\ndef make_line(kind, font, max_width, generator):\n    tokens = [make_token(kind, generator)]\n    while True:\n        candidate = tokens + [make_token(kind, generator)]\n        if font.getlength(\" \".join(candidate)) > max_width:\n            return \" \".join(tokens)\n        tokens = candidate\n\n\ndef render_text_image(font_name, font_size, kind, generator):\n    # BASIC layout: Pillow uses RAQM by default when libraqm is installed, which changes the pixels and the line breaks.\n    font = ImageFont.truetype(str(FONT_DIR / font_name), font_size, layout_engine=ImageFont.Layout.BASIC)\n    image = Image.new(\"RGB\", (IMAGE_SIZE, IMAGE_SIZE), \"white\")\n    draw = ImageDraw.Draw(image)\n\n    line_height = int(font_size * LINE_SPACING)\n    max_width = IMAGE_SIZE - 2 * MARGIN\n    lines = []\n    top = MARGIN\n    while top + line_height <= IMAGE_SIZE - MARGIN:\n        line = make_line(kind, font, max_width, generator)\n        draw.text((MARGIN, top), line, font=font, fill=\"black\")\n        lines.append(line)\n        top += line_height\n\n    return image, \"\\n\".join(lines)\n\n\ndef render_text_set(repeats, seed):\n    generator = random.Random(seed)\n    samples = []\n    for font_name in FONTS:\n        for font_size in FONT_SIZES:\n            for kind in KINDS:\n                for _ in range(repeats):\n                    image, text = render_text_image(font_name, font_size, kind, generator)\n                    samples.append({\n                        \"image\": image,\n                        \"text\": text,\n                        \"font\": font_name,\n                        \"font_size\": font_size,\n                        \"kind\": kind,\n                    })\n    return samples\n",
"code/models/rrdbnet.py": "import torch\nimport torch.nn.functional as F\nfrom torch import nn\n\n\nclass ResidualDenseBlock(nn.Module):\n    def __init__(self, num_feat=64, num_grow_ch=32):\n        super().__init__()\n        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)\n        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)\n        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)\n        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)\n        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_feat, 3, 1, 1)\n        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)\n\n    def forward(self, x):\n        x1 = self.lrelu(self.conv1(x))\n        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))\n        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))\n        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))\n        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))\n        return x5 * 0.2 + x\n\n\nclass RRDB(nn.Module):\n    def __init__(self, num_feat, num_grow_ch=32):\n        super().__init__()\n        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)\n        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)\n        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)\n\n    def forward(self, x):\n        out = self.rdb3(self.rdb2(self.rdb1(x)))\n        return out * 0.2 + x\n\n\nclass RRDBNet(nn.Module):\n    def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32):\n        super().__init__()\n        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)\n        self.body = nn.Sequential(*[RRDB(num_feat, num_grow_ch) for _ in range(num_block)])\n        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)\n        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)\n        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)\n        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)\n        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)\n        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)\n\n    def forward(self, x):\n        feat = self.conv_first(x)\n        feat = feat + self.conv_body(self.body(feat))\n        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode=\"nearest\")))\n        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode=\"nearest\")))\n        return self.conv_last(self.lrelu(self.conv_hr(feat)))\n",
"code/models/resshift.py": "from __future__ import annotations\n\nimport math\nimport sys\nfrom contextlib import nullcontext\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nsys.path.append(str(PROJECT_ROOT))\nimport paths\n\nREPO = paths.THIRD_PARTY_ROOT / \"ResShift\"\nCONFIGS = {\"v3\": \"realsr_swinunet_realesrgan256_journal.yaml\"}\nSCALE = 4\n\n\ndef _import_official_code():\n    if not REPO.exists():\n        raise FileNotFoundError(\n            f\"Official ResShift code was not found at {REPO}. \"\n            \"Run: bash code/models/setup_resshift.sh\"\n        )\n    if str(REPO) not in sys.path:\n        sys.path.insert(0, str(REPO))\n    from omegaconf import OmegaConf\n    from utils import util_common, util_net\n    return OmegaConf, util_common, util_net\n\n\ndef _load_checkpoint(path: Path) -> dict:\n    if not path.exists():\n        raise FileNotFoundError(\n            f\"Missing ResShift weight: {path}. Run: \"\n            \"python code/datasets/download_data.py\"\n        )\n    state = torch.load(path, map_location=\"cpu\", weights_only=True)\n    return state.get(\"state_dict\", state)\n\n\nclass ResShift:\n\n    def __init__(\n        self,\n        device: torch.device,\n        weights_dir: Path,\n        version: str = \"v3\",\n        amp: bool = True,\n        seed: int = 0,\n    ):\n        self.device = device\n        self.scale = SCALE\n        OmegaConf, util_common, util_net = _import_official_code()\n        if version not in CONFIGS:\n            raise ValueError(f\"Unsupported ResShift version {version!r}; choose from {tuple(CONFIGS)}\")\n\n        cfg = OmegaConf.load(REPO / \"configs\" / CONFIGS[version])\n        cfg.diffusion.params.sf = self.scale\n        self.diffusion = util_common.instantiate_from_config(cfg.diffusion)\n\n        if device.type == \"mps\":\n            for key, value in vars(self.diffusion).items():\n                if isinstance(value, np.ndarray) and value.dtype == np.float64:\n                    setattr(self.diffusion, key, value.astype(np.float32))\n\n        model = util_common.instantiate_from_config(cfg.model)\n        util_net.reload_model(\n            model,\n            _load_checkpoint(weights_dir / f\"resshift_realsrx4_s4_{version}.pth\"),\n        )\n        self.model = model.eval().to(device)\n\n        autoencoder = util_common.get_obj_from_str(cfg.autoencoder.target)(**cfg.autoencoder.params)\n        util_net.reload_model(\n            autoencoder,\n            _load_checkpoint(weights_dir / \"autoencoder_vq_f4.pth\"),\n        )\n        self.autoencoder = autoencoder.eval().to(device)\n\n        for parameter in list(self.model.parameters()) + list(self.autoencoder.parameters()):\n            parameter.requires_grad_(False)\n\n        self.pad_to = int(cfg.model.params.get(\"lq_size\", 64))\n        self.seed = int(seed)\n        self.amp = bool(amp and device.type in (\"cuda\", \"mps\"))\n\n    def _autocast(self):\n        if not self.amp:\n            return nullcontext()\n        return torch.autocast(device_type=self.device.type, dtype=torch.float16)\n\n    @torch.inference_mode()\n    def upscale(self, lr: np.ndarray, seed: int | None = None) -> np.ndarray:\n        if lr.ndim != 3 or lr.shape[2] != 3:\n            raise ValueError(f\"Expected an HxWx3 RGB image, got shape {lr.shape}\")\n        torch.manual_seed(self.seed if seed is None else int(seed))\n        y = torch.from_numpy(np.ascontiguousarray(lr)).permute(2, 0, 1)[None]\n        y = y.float().div_(255).to(self.device).mul_(2).sub_(1)  \n\n        height, width = y.shape[2:]\n        pad_h = math.ceil(height / self.pad_to) * self.pad_to - height\n        pad_w = math.ceil(width / self.pad_to) * self.pad_to - width\n        if pad_h or pad_w:\n            y = F.pad(y, (0, pad_w, 0, pad_h), mode=\"reflect\")\n\n        with self._autocast():\n            output = self.diffusion.p_sample_loop(\n                y=y,\n                model=self.model,\n                first_stage_model=self.autoencoder,\n                noise=None,\n                noise_repeat=False,\n                clip_denoised=False,\n                denoised_fn=None,\n                model_kwargs={\"lq\": y},\n                progress=False,\n            )\n\n        output = output[:, :, : height * self.scale, : width * self.scale]\n        output = output.float().clamp(-1, 1).mul_(0.5).add_(0.5)\n        pixels = output[0].permute(1, 2, 0).cpu().numpy()\n        return (pixels * 255.0).round().clip(0, 255).astype(np.uint8)\n",
"code/models/upscale.py": "import argparse\nimport gc\nimport json\nimport platform\nimport random\nimport sys\nimport time\nimport zlib\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom tqdm import tqdm\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nsys.path.append(str(PROJECT_ROOT))\nsys.path.insert(0, str(PROJECT_ROOT / \"code\" / \"datasets\"))\nimport paths\nfrom prepare_data import JPEG_QUALITIES, LR_DIR, PROCESSED_DIR, SCALE\nfrom rrdbnet import RRDBNet\nfrom resshift import ResShift\n\nCONFIG = paths.load_config()\nMODELS_DIR = paths.WEIGHTS_ROOT\nOUTPUTS_DIR = paths.OUTPUTS_ROOT\nRESULTS_DIR = paths.RESULTS_ROOT\nRUNTIME_DIR = RESULTS_DIR / \"runtime\"\n\nMETHODS = [\"bicubic\", \"realesrnet\", \"realesrgan\", \"resshift\"]\n# Real-ESRNet and Real-ESRGAN share the RRDBNet generator and training data; only the loss differs.\nWEIGHTS = dict(CONFIG[\"rrdbnet\"][\"weights\"])\nMODEL_CONFIG = {\n    \"num_feat\": 64,\n    \"num_block\": 23,\n    \"num_grow_ch\": 32,\n}\nRESSHIFT_CONFIG = {\n    \"version\": CONFIG[\"resshift\"][\"version\"],\n    \"model_weights\": CONFIG[\"resshift\"][\"model_weights\"],\n    \"autoencoder_weights\": CONFIG[\"resshift\"][\"autoencoder_weights\"],\n    \"amp\": CONFIG[\"resshift\"][\"amp\"],\n    \"seed\": 0,\n}\nWARMUP_IMAGES = CONFIG[\"runtime\"][\"warmup_runs\"]\nSEED_MASK = 0x7FFFFFFF\nMODEL_CODE = {\n    \"bicubic\": [],\n    \"realesrnet\": [\"code/models/rrdbnet.py\"],\n    \"realesrgan\": [\"code/models/rrdbnet.py\"],\n    \"resshift\": [\"code/models/resshift.py\"],\n}\n\n\ndef pick_device():\n    if torch.cuda.is_available():\n        return torch.device(\"cuda\")\n    if torch.backends.mps.is_available():\n        return torch.device(\"mps\")\n    return torch.device(\"cpu\")\n\n\ndef describe_device(device):\n    if device.type == \"cuda\":\n        return torch.cuda.get_device_name(0)\n    if device.type == \"mps\":\n        return f\"Apple GPU (MPS), {platform.processor()}\"\n    return f\"CPU, {platform.processor() or platform.machine()}\"\n\n\ndef device_tag(device):\n    if device.type == \"cuda\":\n        return \"t4\" if \"T4\" in torch.cuda.get_device_name(0) else \"cuda\"\n    return device.type\n\n\ndef synchronize(device):\n    if device.type == \"cuda\":\n        torch.cuda.synchronize()\n    elif device.type == \"mps\":\n        torch.mps.synchronize()\n\n\ndef method_folder(method, seed_offset=0):\n    \"\"\"Offset 0 is the main run; the other ResShift seeds go to resshift_seed<k>.\"\"\"\n    return method if method != \"resshift\" or seed_offset == 0 else f\"resshift_seed{seed_offset}\"\n\n\ndef resshift_seed(image_id, offset=0):\n    return (zlib.crc32(image_id.encode(\"utf-8\")) + offset) & SEED_MASK\n\n\ndef weight_files(method):\n    if method in WEIGHTS:\n        return [WEIGHTS[method]]\n    if method == \"resshift\":\n        return [RESSHIFT_CONFIG[\"model_weights\"], RESSHIFT_CONFIG[\"autoencoder_weights\"]]\n    return []\n\n\ndef cache_key(method, seed_offset):\n    \"\"\"Everything that determines the output pixels apart from the input file itself.\"\"\"\n    key = {\n        \"method\": method,\n        \"weights_sha256\": {name: paths.sha256_file(MODELS_DIR / name) for name in weight_files(method)},\n        \"model_code_sha256\": {name: paths.sha256_file(PROJECT_ROOT / name) for name in MODEL_CODE[method]},\n    }\n    if method in WEIGHTS:\n        key[\"settings\"] = {\"model_config\": MODEL_CONFIG, \"precision\": \"float32\"}\n    if method == \"resshift\":\n        key[\"settings\"] = {\n            \"version\": RESSHIFT_CONFIG[\"version\"],\n            \"official_commit\": CONFIG[\"resshift\"][\"official_commit\"],\n            \"amp\": RESSHIFT_CONFIG[\"amp\"],\n            \"seed_offset\": seed_offset,\n            \"seed_policy\": CONFIG[\"resshift\"][\"seed_policy\"],\n        }\n    return key\n\n\ndef read_manifest(output_root):\n    path = output_root / \"manifest.json\"\n    return json.loads(path.read_text()) if path.exists() else None\n\n\ndef cached(output_root, key, inputs):\n    \"\"\"True when the saved outputs were made with the same key from the same input files and are unchanged.\"\"\"\n    manifest = read_manifest(output_root)\n    if manifest is None or manifest[\"key\"] != key:\n        return False\n    for name, input_hash in inputs.items():\n        entry = manifest[\"files\"].get(name)\n        if entry is None or entry[\"input_sha256\"] != input_hash:\n            return False\n        if paths.sha256_file(output_root / name) != entry[\"output_sha256\"]:\n            return False\n    return True\n\n\ndef load_rrdbnet(method, device):\n    network = RRDBNet(\n        num_feat=MODEL_CONFIG[\"num_feat\"],\n        num_block=MODEL_CONFIG[\"num_block\"],\n        num_grow_ch=MODEL_CONFIG[\"num_grow_ch\"],\n    )\n    checkpoint = torch.load(MODELS_DIR / WEIGHTS[method], map_location=\"cpu\", weights_only=True)\n    state = checkpoint.get(\"params_ema\", checkpoint.get(\"params\", checkpoint))\n    network.load_state_dict(state, strict=True)\n    return network.eval().to(device)\n\ndef load_resshift(device):\n    return ResShift(\n        device=device,\n        weights_dir=MODELS_DIR,\n        version=RESSHIFT_CONFIG[\"version\"],\n        amp=RESSHIFT_CONFIG[\"amp\"],\n        seed=RESSHIFT_CONFIG[\"seed\"],\n    )\n\ndef upscale_bicubic(image):\n    size = (image.width * SCALE, image.height * SCALE)\n    return image.resize(size, Image.BICUBIC)\n\n\n@torch.inference_mode()\ndef upscale_rrdbnet(image, network, device):\n    pixels = np.asarray(image, dtype=np.float32) / 255.0\n    batch = torch.from_numpy(pixels).permute(2, 0, 1).unsqueeze(0).to(device)\n    output = network(batch).clamp(0, 1)\n    synchronize(device)\n    output = output.squeeze(0).permute(1, 2, 0).cpu().numpy()\n    return Image.fromarray((output * 255.0).round().astype(np.uint8))\n\n\ndef run_method(method, image_ids, split, device, seed_offset=0, inputs=None, force=False):\n    if method == \"bicubic\":\n        device = torch.device(\"cpu\")\n    folder = method_folder(method, seed_offset)\n    output_root = OUTPUTS_DIR / split / folder\n    inputs = inputs or {}\n    wanted = {f\"jpeg_{quality}/{image_id}.png\": inputs[f\"lr/jpeg_{quality}/{image_id}.png\"]\n              for quality in JPEG_QUALITIES for image_id in image_ids}\n    key = cache_key(method, seed_offset)\n    if not force and cached(output_root, key, wanted):\n        print(f\"{folder}: outputs match manifest.json (same weights, code, settings and inputs); skipped\")\n        return {\"cached\": True}\n    if read_manifest(output_root) is not None:\n        print(f\"{folder}: manifest.json does not match the current weights, code, settings or inputs; recomputing\")\n\n    wall_start = time.perf_counter()\n    if device.type == \"cuda\":\n        gc.collect()\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n    if method in WEIGHTS:\n        network = load_rrdbnet(method, device)\n    elif method == \"resshift\":\n        network = load_resshift(device)\n    else:\n        network = None\n\n    def upscale(image, image_id=None):\n        if method == \"bicubic\":\n            return upscale_bicubic(image)\n        if method == \"resshift\":\n            output = network.upscale(np.asarray(image.convert(\"RGB\")), seed=resshift_seed(image_id, seed_offset))\n            return Image.fromarray(output)\n        return upscale_rrdbnet(image, network, device)\n\n    warmup_image = Image.open(LR_DIR / \"jpeg_none\" / f\"{image_ids[0]}.png\").convert(\"RGB\")\n    for _ in range(WARMUP_IMAGES):\n        upscale(warmup_image, image_ids[0])\n\n    seconds = []\n    files = {}\n    for quality in JPEG_QUALITIES:\n        output_dir = output_root / f\"jpeg_{quality}\"\n        output_dir.mkdir(parents=True, exist_ok=True)\n        for image_id in tqdm(image_ids, desc=f\"{folder} jpeg_{quality}\"):\n            image = Image.open(LR_DIR / f\"jpeg_{quality}\" / f\"{image_id}.png\").convert(\"RGB\")\n            synchronize(device)\n            start = time.perf_counter()\n            result = upscale(image, image_id)\n            synchronize(device)\n            seconds.append({\"image_id\": image_id, \"jpeg\": str(quality), \"seconds\": time.perf_counter() - start})\n            name = f\"jpeg_{quality}/{image_id}.png\"\n            result.save(output_root / name)\n            files[name] = {\"input_sha256\": wanted[name], \"output_sha256\": paths.sha256_file(output_root / name)}\n\n    report = {\n        \"images\": len(seconds),\n        \"device\": describe_device(device),\n        \"seconds_per_image_mean\": float(np.mean([row[\"seconds\"] for row in seconds])),\n        \"seconds_per_image_median\": float(np.median([row[\"seconds\"] for row in seconds])),\n        \"wall_seconds\": time.perf_counter() - wall_start,\n        \"warmup_runs\": WARMUP_IMAGES,\n    }\n    if device.type == \"cuda\":\n        report[\"peak_memory_allocated_mib\"] = torch.cuda.max_memory_allocated() / 2**20\n        report[\"peak_memory_reserved_mib\"] = torch.cuda.max_memory_reserved() / 2**20\n    manifest = {\"split\": split, \"folder\": folder, \"key\": key, \"device\": describe_device(device), \"files\": files}\n    (output_root / \"manifest.json\").write_text(json.dumps(manifest, indent=1))\n    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame(seconds).to_csv(RUNTIME_DIR / f\"timing_{split}_{folder}_{device_tag(device)}.csv\", index=False)\n    del network\n    return report\n\n\ndef main(split=\"dev\", methods=METHODS, seed_offsets=(0,), force=False):\n    manifest = pd.read_csv(PROCESSED_DIR / \"manifest.csv\")\n    image_ids = manifest.loc[manifest[\"split\"] == split, \"image_id\"].tolist()\n    inputs = json.loads((PROCESSED_DIR / \"inputs_sha256.json\").read_text())\n    device = pick_device()\n    # Same global state as the stage 4 ResShift notebook; ResShift also reseeds torch per image.\n    random.seed(CONFIG[\"data\"][\"seed\"])\n    np.random.seed(CONFIG[\"data\"][\"seed\"])\n    torch.manual_seed(CONFIG[\"data\"][\"seed\"])\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n    report = {\n        \"split\": split,\n        \"device\": describe_device(device),\n        \"torch\": torch.__version__,\n        \"cuda\": torch.version.cuda,\n        \"python\": sys.version.split()[0],\n        \"precision\": {\"bicubic\": \"uint8 (Pillow, CPU)\", \"rrdbnet\": \"float32\", \"resshift\": CONFIG[\"resshift\"][\"precision\"]},\n        \"batch_size\": 1,\n        \"input_size\": 512 // SCALE,\n        \"output_size\": 512,\n        \"rrdbnet_config\": MODEL_CONFIG,\n        \"rrdbnet_weights\": WEIGHTS,\n        \"rrdbnet_parameters\": sum(p.numel() for p in RRDBNet().parameters()),\n        \"resshift_config\": RESSHIFT_CONFIG,\n        \"config\": paths.config_path().name,\n        \"config_sha256\": paths.sha256_file(paths.config_path()),\n        \"methods\": {},\n    }\n    split_start = time.perf_counter()\n    for method in methods:\n        for offset in (seed_offsets if method == \"resshift\" else (0,)):\n            report[\"methods\"][method_folder(method, offset)] = run_method(method, image_ids, split, device, offset, inputs, force)\n    report[\"split_wall_seconds\"] = time.perf_counter() - split_start\n    if all(entry.get(\"cached\") for entry in report[\"methods\"].values()):\n        return\n\n    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)\n    tag = \"_\".join(sorted(report[\"methods\"]))\n    report_json = json.dumps(report, indent=2)\n    (RUNTIME_DIR / f\"upscale_report_{split}_{device_tag(device)}_{tag}.json\").write_text(report_json)\n    print(report_json)\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--split\", default=\"dev\", choices=[\"dev\", \"test\"])\n    parser.add_argument(\"--methods\", nargs=\"+\", default=METHODS, choices=METHODS)\n    parser.add_argument(\"--seed-offsets\", nargs=\"+\", type=int, default=[CONFIG[\"resshift\"][\"seed_offset\"]])\n    parser.add_argument(\"--force\", action=\"store_true\", help=\"recompute even when the outputs match their manifest\")\n    args = parser.parse_args()\n    main(args.split, args.methods, args.seed_offsets, args.force)\n",
"code/models/setup_resshift.sh": "#!/usr/bin/env bash\nset -euo pipefail\ncd \"$(dirname \"$0\")/../..\"\nPY=\"${PYTHON:-python}\"\nREPO=\"$(\"$PY\" -c 'import paths; print(paths.THIRD_PARTY_ROOT)')/ResShift\"\nCOMMIT=\"bb03b7d21614cace01787e097c8a6ab6b945227d\"\n\nif [ ! -d \"$REPO/.git\" ]; then\n    mkdir -p \"$(dirname \"$REPO\")\"\n    git clone https://github.com/zsyOAOA/ResShift.git \"$REPO\"\nfi\n\ngit -C \"$REPO\" fetch --depth 1 origin \"$COMMIT\" 2>/dev/null || true\ngit -C \"$REPO\" checkout -q \"$COMMIT\"\necho \"Official ResShift checked out at $(git -C \"$REPO\" rev-parse --short HEAD)\"\n"
}
HASHES = {
"paths.py": "c1a1e48b3c0c17e44368392f4f4b089748b619191ea67bb3f05fedb90564bebe",
"inference.py": "2f3636901947938dfd1834fcd3fc20a75f0d433c9fd2bc81e36e42935e9445c8",
"configs/default.yaml": "be3145a95183f1c305ae07b829a4895791aa4c622f4cb942528bc34cb3ea7383",
"configs/final.yaml": "70f6c341e963a068e901be19d9dc11f36cc5df078ef04dfaf9df0846abbb1059",
"configs/final.yaml.sha256": "70e0c5cb423b81a130314fccfd69a8afab1538e89c403b1ea367b245142bfdc0",
"scripts/SHA256SUMS": "3c4dbeaa21bb5cc764032b9d41b6fca4f1dcf2382a873bdb2ff8d74766507743",
"code/datasets/download_data.py": "c7329a10824b4a19b88972bb18cc9d975e6b12da789e1b7026b4c3668888444d",
"code/datasets/prepare_data.py": "2968ad613e34bfeab2da559cd10470f33904a13d241b0aa2f6e0d3b2b32ff08e",
"code/datasets/render_text.py": "7549c21e35066efe829ab741b42a64389d356caef21e695eb528f332876beffa",
"code/models/rrdbnet.py": "2d9f0ddb8334df222ccb05a90a5ac5faf4602369361bf9154f83ccf8643784f4",
"code/models/resshift.py": "17cddf4f507dec3582c90a06039f1be92f6c0e61e74208776536ec26e99d5ed1",
"code/models/upscale.py": "86441b857b72b8357102055ba6fed617f4b6baa847c6c5fbb9d7f38ea5f9b673",
"code/models/setup_resshift.sh": "ebe6d411f130197a9de2ac1f613c6de29c6f09ba66728fea930a808736b543e2"
}

for name, text in FILES.items():
    path = WORK / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == HASHES[name], f'{name} differs from the build: {actual}'
print(len(FILES), 'repository files written and verified')


In [ ]:
from importlib.metadata import version
constraints = WORK / 'constraints.txt'
constraints.write_text(f"torch=={version('torch')}\ntorchvision=={version('torchvision')}\n")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraints), *['omegaconf==2.3.0', 'einops==0.8.2', 'timm==1.0.30', 'addict==2.4.0', 'future==1.0.0', 'loralib==0.1.2', 'PyYAML==6.0.3']], check=True)


In [ ]:
import csv
import json
import shutil
if archive.is_file():
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(WORK)
else:
    shutil.copytree(archive / 'data', WORK / 'data', dirs_exist_ok=True)
hashes_file = WORK / 'data/processed/inputs_sha256.json'
assert hashlib.sha256(hashes_file.read_bytes()).hexdigest() == INPUTS_SHA256_JSON, 'inputs_sha256.json differs from the build'
expected = json.loads(hashes_file.read_text())
lr = sorted((WORK / 'data/processed/lr').rglob('*.png'))
for path in lr:
    name = str(path.relative_to(WORK / 'data/processed'))
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected[name], f'{name} differs from inputs_sha256.json'
with open(WORK / 'data/processed/manifest.csv') as handle:
    images = sum(row['split'] == SPLIT for row in csv.DictReader(handle))
assert len(lr) == 4 * images, f'{len(lr)} inputs found, {4 * images} expected for {SPLIT}'
print(len(lr), 'inputs verified')


In [ ]:
subprocess.run([sys.executable, 'code/datasets/download_data.py', '--only', 'weights'], check=True)   # verifies scripts/SHA256SUMS
subprocess.run(['bash', 'code/models/setup_resshift.sh'], check=True, env=dict(os.environ, PYTHON=sys.executable))


In [ ]:
command = [sys.executable, 'inference.py', '--config', 'configs/final.yaml', '--split', SPLIT, '--methods', *METHODS,
           '--seed-offsets', *map(str, SEED_OFFSETS)]
print(' '.join(command))
subprocess.run(command, check=True)   # inference.py refuses the test split unless configs/final.yaml matches final.yaml.sha256


In [ ]:
runtime = WORK / 'results/runtime'
runtime.mkdir(parents=True, exist_ok=True)
environment = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
(runtime / f'environment_{NOTEBOOK}.txt').write_text(f'python {sys.version}\ntorch {torch.__version__} cuda {torch.version.cuda}\n\n{gpu}\n{environment}')

out = (Path('/kaggle/working') if ON_KAGGLE else Path('/content')) / f'{NOTEBOOK}_results.zip'
with zipfile.ZipFile(out, 'w', compression=zipfile.ZIP_DEFLATED) as handle:
    for folder in sorted((WORK / 'outputs' / SPLIT).iterdir()):
        for path in sorted(folder.rglob('*')):
            if path.is_file():
                handle.write(path, path.relative_to(WORK))
    for path in sorted(runtime.iterdir()):
        handle.write(path, path.relative_to(WORK))
print(out, round(out.stat().st_size / 2**20, 1), 'MiB')
if ON_KAGGLE:
    from IPython.display import FileLink, display
    os.chdir('/kaggle/working')
    display(FileLink(out.name))   # click to download; after "Save Version" it is also under Output
else:
    from google.colab import files
    files.download(str(out))
